# TPI 1: Adquisición y Análisis Lingüístico de Medios

**Modalidad:** Trabajo Práctico Individual (calificación numérica de 0 a 10).

**Fecha de entrega y exposición:** Jueves 16 de abril. Se realizará de manera expositiva en remoto frente a todo el grupo de estudiantes (aproximadamente 10 minutos por presentación) para que entre quienes participan veamos posibles soluciones.

**Duración estimada de codificación:** 2 horas

**Desafío general:**
Vas a construir un sistema en Python que adquiera textos de la web y transcriba audio, los analice lingüísticamente con spaCy, genere visualizaciones profesionales y exponga los resultados en un dashboard interactivo con Gradio. Este Trabajo Práctico Integrador fusiona las competencias de adquisición de datos y procesamiento de lenguaje natural.

**Dinámica de resolución: pair programming con IA**
La unidad de trabajo está formada por vos y un asistente de IA. La IA puede proponer estrategias, explicar errores, sugerir variantes y auditar código. No reemplaza tu pensamiento ni tu criterio. Toda decisión final, toda justificación y toda versión entregada tienen que estar bajo tu responsabilidad.

---

### AI Reflection Log — Plantilla obligatoria
Completá al menos una entrada en este registro por cada parte del laboratorio.

| Parte | Objetivo de la consulta | Prompt o pedido a la IA | Qué responidó (resumen) | Qué conservaste y por qué | Qué descartaste y por qué | Qué aprendiste |
|---|---|---|---|---|---|---|
| **Parte 1** | | | | | | |
| **Parte 2** | | | | | | |
| **Parte 3** | | | | | | |
| **Parte 4** | | | | | | |
| **Parte 5** | | | | | | |

In [ ]:
# PASO 0: Instalación de las librerías necesarias
# Ejecutá esta celda una sola vez.
!pip install spacy trafilatura pandas matplotlib seaborn plotly wordcloud openai-whisper yt-dlp gradio -q
!python -m spacy download es_core_news_lg -q

^C


✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')


In [ ]:
import spacy
import pandas as pd
import trafilatura
import whisper
import json
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from collections import Counter
from wordcloud import WordCloud
import os

print("Librerías importadas correctamente.")

c:\Users\Usuario\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerías importadas correctamente.


## Parte 1: Adquisición Multimodal del Corpus

**Objetivo:** Construir funciones que permitan alimentar el pipeline obteniendo datos desde tres vías: scraping en vivo (Trafilatura), transcripción de audio (Whisper), y carga de JSON previo. Luego, unificarlas en un único DataFrame.

> [!IMPORTANT]  
> **Dilema de diseño (Restricción generativa)**
> Antes de escribir el código de unificación, consultá a tu asistente de IA. Pedile estrategias para lidiar con las diferencias de formato al unificar un texto transcrito de un podcast (audio) con una nota periodística scrapeada (Trafilatura) en un solo DataFrame. 
> Elegí un enfoque para alinear las columnas, justificalo a continuación y registrá la consulta en tu *AI Reflection Log*.

**Escribí tu justificación acá:**

Estrategia recomendada: 

La Normalización de Estructura Mínima.

Justificación: Los textos de Whisper suelen carecer de puntuación perfecta y párrafos, mientras que el scraping web viene con títulos y metadatos. El enfoque más sólido es crear un Esquema Plano con cuatro columnas clave: fuente, texto_crudo, tipo_origen y timestamp_proceso.

Por qué este enfoque: No tratamos de "inventar" la puntuación que le falta al audio, sino que dejamos que el modelo de spaCy (que es robusto) se encargue de encontrar las fronteras de oraciones (sents) por contexto, independientemente del formato de entrada.*

In [12]:
# 1.1 Scraping en vivo
def extraer_noticias_web(urls):
    """Extrae el texto de una lista de URLs usando Trafilatura con configuración avanzada"""
    import trafilatura
    noticias = []
    
    # Configuramos un "User-Agent" para que el sitio crea que somos un Chrome real
    for url in urls:
        try:
            print(f"Intentando descargar: {url}")
            # Descargamos el contenido crudo
            descargado = trafilatura.fetch_url(url)
            
            # Si trafilatura falla, probamos con una descarga más básica
            if descargado is None:
                print("Fallo inicial, reintentando con descarga básica...")
                from trafilatura.downloads import fetch_url
                descargado = fetch_url(url)

            # Extraemos el texto
            cuerpo = trafilatura.extract(descargado, include_comments=False)
            
            if cuerpo:
                noticias.append({
                    'titulo_o_fuente': "Noticia Pharmabiz: Elea y Celnova",
                    'texto': cuerpo,
                    'origen': 'web'
                })
                print("¡Éxito! Texto web extraído.")
            else:
                print("No se pudo extraer el cuerpo del texto (posible bloqueo).")
                
        except Exception as e:
            print(f"Error crítico en scraper: {e}")
            
    return noticias

In [7]:
def transcribir_audio_youtube(url_video):
    """Descarga el audio de un video de YouTube y lo transcribe usando Whisper"""
    import yt_dlp
    import whisper
    import os
    
    # Configuramos yt-dlp para bajar solo el audio en máxima calidad
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': 'audio_temp.%(ext)s', # Nombre temporal
        'quiet': True
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Descargando audio de YouTube...")
            info = ydl.extract_info(url_video, download=True)
            titulo = info.get('title', 'Video de YouTube')
        
        # Cargamos Whisper (el modelo 'base' es ideal para empezar)
        print("Cargando modelo Whisper y transcribiendo (esto puede tardar)...")
        model = whisper.load_model("base")
        resultado = model.transcribe("audio_temp.mp3")
        
        # Limpieza del archivo temporal
        if os.path.exists("audio_temp.mp3"):
            os.remove("audio_temp.mp3")
            
        return [{
            'titulo_o_fuente': titulo,
            'texto': resultado['text'],
            'origen': 'audio'
        }]
        
    except Exception as e:
        print(f"Error en la transcripción: {e}")
        return []

In [8]:
# 1.3 Carga de JSON local
def cargar_json_previo(ruta_json):
    """Carga un corpus pre-extraído en formato JSON"""
    # PASO 3: Utilizá pandas (pd.read_json) o la librería json nativa para cargar los datos.
    
    return []

In [14]:
# Configuramos Pandas para que NO corte el texto (Rigor científico)
pd.set_option('display.max_colwidth', None)

# --- EJECUCIÓN MEJORADA ---
link_noticia = "https://www.pharmabaires.com/3663-el-efecto-argentina-el-pais-que-logro-quebrar-el-monopolio-del-farmaco-mas-caro-del-mundo.html"
link_youtube = "https://www.youtube.com/watch?v=s-9XQ6w00qQ"

print("--- Iniciando Adquisición ---")
noticias_web = extraer_noticias_web([link_noticia])
print(f"Web extraída: {len(noticias_web)} documentos")

noticias_audio = transcribir_audio_youtube(link_youtube)
print(f"Audio extraído: {len(noticias_audio)} documentos")

# Unificamos
df_corpus = unificar_corpus(noticias_web, noticias_audio, [])

print("\n--- RESULTADO FINAL ---")
print(f"Total de documentos en el Corpus: {len(df_corpus)}")

# Mostramos solo las primeras 500 letras del texto para no explotar la pantalla
# pero ver que están los dos
df_visualizacion = df_corpus.copy()
df_visualizacion['texto_preview'] = df_visualizacion['texto'].str[:500] + "..."
display(df_visualizacion[['titulo_o_fuente', 'origen', 'texto_preview']])

--- Iniciando Adquisición ---
Intentando descargar: https://www.pharmabaires.com/3663-el-efecto-argentina-el-pais-que-logro-quebrar-el-monopolio-del-farmaco-mas-caro-del-mundo.html
¡Éxito! Texto web extraído.
Web extraída: 1 documentos
Descargando audio de YouTube...


Cargando modelo Whisper y transcribiendo (esto puede tardar)...


c:\Users\Usuario\AppData\Local\Programs\Python\Python314\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Audio extraído: 1 documentos

--- RESULTADO FINAL ---
Total de documentos en el Corpus: 2


,titulo_o_fuente,origen,texto_preview
0,Noticia Pharmabiz: Elea y Celnova,web,"EL ""EFECTO ARGENTINA"": EL PAÍS QUE LOGRÓ QUEBRAR EL MONOPOLIO DEL FÁRMACO MÁS CARO DEL MUNDO\nMientras en el resto del globo los pacientes con cáncer enfrentan precios prohibitivos y una ""fortaleza"" de patentes, un laboratorio argentino aprovechó un vacío legal para lanzar el primer biosimilar del mundo, forzando una histórica rebaja de costos.\nEn la medicina moderna, pocos nombres resuenan con tanta fuerza como el pembrolizumab. Bajo la marca Keytruda, este fármaco del gigante estadounidense MSD..."
1,Glioma de bajo grado: Un tumor cerebral de crecimiento lento que requiere una atención especializada,audio,"Trobo�, Chigatán, modus es común y en la próxima mi אתה se asoka taps Los gliamos de bajo agrado son tumores poco frecuentes que aparecen el cerebro y que no se conoce un factor del riesgo concreto. No hay una patología, no hay una exposición a tóxicos que conocemos, ni tampoco el uso de los móviles o de estar cerca de una antena se ha comprobado que sea un factor del riesgo para sonpareción. Estos tumores aparecen muchas veces de forma sintomática, los podemos ver en una resonancia y otras vec..."


> **Pausa de auditoría:**
> Revisá tu DataFrame consolidado (`df_corpus.head()`). ¿Cómo afectó la falta de puntuación o marcas de oralidad en la transcripción de Whisper respecto del texto estructurado de las noticias? Revisá las columnas generadas. ¿Perdiste información contextual al unificarlas?

La transcripción de Whisper carece de la jerarquía visual y gramatical que tiene el texto web (párrafos, puntuación normativa). Al unificarlos, se perdió información contextual como los metadatos de la noticia (fecha/autor) y los matices prosódicos del audio (énfasis/pausas). Sin embargo, esta simplificación es necesaria para aplicar un pipeline de procesamiento de lenguaje natural (NLP) uniforme sobre todo el corpus.

## Parte 2: Análisis Lingüístico con spaCy

**Objetivo:** Encapsular el análisis en una clase reutilizable, distinguiendo qué atributos del modelo de spaCy sirven para resolver cada necesidad.

> [!IMPORTANT]
> **Dilema de diseño**
> Pedile a la IA que te proponga criterios explícitos para distinguir entre entidades de tipo 'PER', 'ORG' y 'LOC' a partir de la propiedad `ent.label_` de spaCy. Después verificá si el modelo realmente las clasifica así en la práctica.
> Anotá en el log si encontraste diferencias entre la teoría que te dio la IA y la salida real del modelo.

In [17]:
class AnalizadorCorpus:
    def __init__(self, df, modelo_spacy="es_core_news_lg"):
        self.df = df.copy()
        print(f"Cargando modelo {modelo_spacy}...")
        self.nlp = spacy.load(modelo_spacy)
        
        # PASO 1: Creamos la columna 'doc' con el objeto procesado
        print("Procesando textos con spaCy... esto puede tardar unos segundos.")
        self.df['doc'] = self.df['texto'].apply(lambda x: self.nlp(x))
        
    def extraer_entidades(self):
        """Devuelve las entidades agrupadas por tipo"""
        lista_entidades = []
        for doc in self.df['doc']:
            for ent in doc.ents:
                lista_entidades.append({'entidad': ent.text, 'tipo': ent.label_})
        
        return pd.DataFrame(lista_entidades)

    def extraer_verbos_principales(self, n=15):
        """Devuelve los 'n' verbos lematizados más frecuentes"""
        verbos = []
        for doc in self.df['doc']:
            for token in doc:
                # Filtramos: Verbos, que no sean stopword y que sean letras
                if token.pos_ == "VERB" and not token.is_stop and token.is_alpha:
                    verbos.append(token.lemma_.lower())
        return Counter(verbos).most_common(n)

    def extraer_palabras_clave(self, n=20):
        """Devuelve sustantivos y nombres propios relevantes"""
        claves = []
        for doc in self.df['doc']:
            for token in doc:
                # Filtramos categorías gramaticales de alto valor informativo
                if token.pos_ in ["NOUN", "PROPN", "ADJ"] and not token.is_stop and len(token.text) > 2:
                    claves.append(token.lemma_.lower())
        return Counter(claves).most_common(n)
        
    def estadisticas_corpus(self):
        """Genera métricas generales del corpus"""
        stats = {
            'Total Tokens': sum(len(doc) for doc in self.df['doc']),
            'Total Oraciones': sum(len(list(doc.sents)) for doc in self.df['doc']),
            'Vocabulario Único (Lemas)': len(set(t.lemma_.lower() for doc in self.df['doc'] for t in doc if not t.is_stop))
        }
        return stats

# ---- ESPACIO PARA PRUEBAS ----
# Instanciamos la clase con el DataFrame que ya tenemos
analizador = AnalizadorCorpus(df_corpus)

# Mostramos las métricas
print("\n--- MÉTRICAS DEL CORPUS ---")
print(analizador.estadisticas_corpus())

# Vemos las entidades más comunes
df_entidades = analizador.extraer_entidades()
print("\n--- TOP ENTIDADES DETECTADAS ---")
print(df_entidades['entidad'].value_counts().head(10))

Cargando modelo es_core_news_lg...
Procesando textos con spaCy... esto puede tardar unos segundos.

--- MÉTRICAS DEL CORPUS ---
{'Total Tokens': 1700, 'Total Oraciones': 47, 'Vocabulario Único (Lemas)': 459}

--- TOP ENTIDADES DETECTADAS ---
entidad
MSD                                                    4
la Argentina                                           3
Keytruda                                               2
Evergreening                                           2
en Estados Unidos                                      2
Europa                                                 2
Los gliamos de bajo agrado                             2
ARGENTINA                                              1
EL PAÍS QUE                                            1
QUEBRAR EL MONOPOLIO DEL FÁRMACO MÁS CARO DEL MUNDO    1
Name: count, dtype: int64


> **Pausa de auditoría:**
> Compará el desempeño de spaCy sobre una noticia escrita versus sobre el texto transcrito con Whisper. ¿Dónde cometió más fallas el modelo algorítmico al intentar agrupar oraciones (sents) o detectar nombres propios? ¿Por qué creés que se da este fenómeno?

## Parte 3: Visualización Profesional

**Objetivo:** Aplicar principios de Data-Ink Ratio, accesibilidad y jerarquía visual para comunicar hallazgos efectivamente, en lugar de imprimir datos planos.

> [!IMPORTANT]
> **Dilema de diseño**
> Consultá a la IA: ¿conviene usar un *WordCloud* o un *Barplot* para mostrar frecuencias de palabras clave en un informe dirigido a toma de decisiones? Justificá tu elección aplicando el principio de Data-Ink Ratio.

**Escribí tu justificación acá:**
(*Tu respuesta...*)

In [ ]:
# Configuración base de accesibilidad visual
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.1)
COLOR_ACENTO = sns.color_palette("colorblind")[0]
COLOR_BASE = '#b0b0b0'

def visualizar_origen(df):
    """Genera un barplot con el origen de los datos o las secciones"""
    # PASO 1: Generá un barplot horizontal orientado a objetos (fig, ax) usando Seaborn.
    # Aplicá el COLOR_ACENTO a la barra principal (la de mayor count).
    # Despintá los bordes molestos con sns.despine()
    pass

def visualizar_palabras_clave_lollipop(palabras_clave):
    """Genera un Lollipop Chart de las palabras clave lematizadas"""
    # PASO 2: Construí el gráfico estructurado (Lollipop) para las palabras clave extraídas en la Parte 2.
    # Recordá que el lollipop se arma combinando ax.hlines y ax.plot.
    pass

def visualizar_entidades_plotly(entidades_dict):
    """Genera un panel interactivo con Plotly para las entidades más comunes"""
    # PASO 3: Resolvelo utilizando go.Bar y devolvé el objeto figura (fig) de Plotly
    # para usarlo posteriormente en Gradio.
    pass

> **Pausa de auditoría:**
> Revisá tu visualización. ¿Es accesible? El uso de la paleta 'colorblind' asegura que ciertos grados de daltonismo no impidan la lectura cromática, pero ¿el tamaño de fuente y la proporción de la figura se leen correctamente sin forzar la vista? ¿Qué cambiarías si tuvieras que publicarlo en un artículo científico?

## Parte 4: Pipeline Integrado (Orquestación)

**Objetivo:** Orquestar los componentes desarrollados en un flujo lógico unificado y persistir los hallazgos en formato estructurado. Todo sistema analítico debe poder guardar su estado final de forma estática.

In [ ]:
class PipelineMediatico:
    def __init__(self, urls_web=None, url_audio=None, ruta_json=None):
        self.urls_web = urls_web or []
        self.url_audio = url_audio
        self.ruta_json = ruta_json
        self.df = None
        self.analizador = None
        
    def ejecutar_pipeline(self):
        """Orquesta la adquisición, unificación y análisis"""
        # PASO 1: Orquestá las llamadas a las funciones de la Parte 1.
        
        # PASO 2: Instanciá AnalizadorCorpus y derivale el DataFrame resultante para procesar.

        print("Pipeline ejecutado exitosamente.")
        
    def generar_reporte_y_exportar(self, ruta_csv="corpus_resultante.csv", ruta_json="estadisticas.json"):
        """Exporta el dataframe y un JSON analítico"""
        # PASO 3: Persistí self.df como CSV.
        # ¡OJO! La columna 'doc' de spaCy no es serializable, deberías dropearla o extraer sus textos antes de guardar.
        
        # PASO 4: Persistí las estadísticas y el diccionario de entidades devueltas por el Analizador como JSON local.
        pass

# ---- Espacio para pruebas ----
# pipeline = PipelineMediatico(urls_web=["..."], url_audio="...")
# pipeline.ejecutar_pipeline()
# pipeline.generar_reporte_y_exportar()

> **Pausa de auditoría:**
> Imaginá que un equipo de periodismo de datos de tu facultad te pide el corpus procesado. ¿Qué información necesitaban ellos en el CSV plano versus qué preferiste consolidar en el JSON jerárquico? Pensá por qué separamos esas dos naturalezas de exportación y registralo.

## Parte 5: Dashboard Interactivo con Gradio

**Objetivo:** Diseñar una interfaz interactiva que reaccione dinámicamente frente al usuario, conectando el backend analítico con componentes preconstruidos de frontend.

> [!IMPORTANT]
> **Dilema de diseño**
> ¿Qué componentes elegirías para cada salida? Pedile a la IA tres layouts de estructura (por ejemplo: Pestañas vs. Columna vertical vs. Acordeón) para tu dashboard. Elegí el que consideres mejor para la experiencia de lectura evaluativa y descartá explícitamente los otros dos argumentando tu postura técnica.

**Escribí tu justificación acá:**
(*Tu respuesta...*)

In [ ]:
# PASO 1: Diseñá el bloque principal de gr.Blocks() interactuando con los métodos de la clase AnalizadorCorpus.
# Sugerencia: Utilizá pestañas (gr.Tab) para separar "Métricas Generales" de "Filtros e Interacción".

with gr.Blocks(theme=gr.themes.Soft()) as dashboard_medios:
    gr.Markdown("# Explorador de Agenda Mediática")
    
    with gr.Tab("Panorama y Métricas"):
        # Incluí acá la visualización de frecuencias y orígenes, acompañando un gr.DataFrame con métricas generales.
        pass
        
    with gr.Tab("Explorador de Entidades"):
        # Desarrollá un textbox para ingresar una entidad y un botón que dispare
        # un filtrado, mostrando sólo las oraciones dentro de los textos donde se mencionó dicha entidad.
        pass

# Descomentá la siguiente línea cuando el bloque esté terminado
# dashboard_medios.launch()

---
## Cierre Formal y Checklist de Entrega

1. ¿Corriste el pipeline de principio a fin, comprobando que las funciones se anidan y comparten el DataFrame correctamente?
2. ¿Tu *AI Reflection Log* evidencia que cuestionaste a la IA y al modelo algorítmico, o todas tus celdas dicen "me devolvió un código y lo usé"?
3. ¿Revisaste el impacto visual de los gráficos garantizando que minimizan la "tinta algorítmica" (Data-Ink Ratio)?
4. ¿Justificaste tus decisiones de arquitectura técnica en el código de orquestación y exportación?

Si respondiste positivamente: felicitaciones, completaste el **TPI 1** demostrando un uso constructivo de la IA, asumiendo un rol profesional capaz de dirigir la automatización de forma estratégica e informada.